# Build the network: OSM + PeMS + Census (Fresno, 25 km²)
Builds `data/Fresno/250327231442`, the network used by examples 02-04. See the README for the PeMS account, Census API key and TAZ shapefile it needs.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Path management
main_dir = str(Path(os.path.abspath("")).parents[0])
os.chdir(main_dir)
print('main dir:', main_dir)

sys.path.append(os.path.join(main_dir, 'src'))

In [ ]:
%load_ext autoreload
# %autoreload 2

from mattwin import osmbuilder

In [ ]:
ts = datetime.now().strftime('%y%m%d%H%M%S')
ts

In [ ]:
#City name for which we want to download the data.
area_of_interest = 'Fresno'
ts='250327231442'
req_format = 'text'
year_station_hr_data=2024 #year1
year_station_metadata=2023 #year2
district_id=6
data_dir=os.path.join(main_dir, 'data')
station_metadata_option=4
station_hr_data_option=8
station_metadata_file='input/d06_text_meta_2023_08_08.txt'
station_hour_data_file='input/d06_text_station_hour_2024_08.txt'
lat=36.7394421
lon=-119.78483
county='Fresno, CA'
width_km=5
height_km=5
counties_shp='input/counties/counties.shp'

In [ ]:
# # Parameters
# area_of_interest = "SantaClara"
# req_format = "text"
# year_station_hr_data = "2023"
# year_station_metadata = "2022"
# district_id = 4
# lat = 37.3544
# lon = -121.9131
# station_metadata_option = "7"
# station_hr_data_option = "10"
# station_metadata_file = "input/d04_text_meta_2022_10_22.txt"
# station_hour_data_file = "input/d04_text_station_hour_2023_10.txt.gz"
# county='Santa Clara, CA'

In [ ]:
# Timestamp is used to set the folder in which the data will be saved

folder_path = os.path.join(data_dir,area_of_interest,ts)
print(folder_path)
for sub in ('input', 'output/area_expansion'):
    os.makedirs(os.path.join(folder_path, sub), exist_ok=True)
if not os.path.exists(os.path.join(folder_path, counties_shp)):
    osmbuilder._credential('CENSUS_API_KEY')  # fail before the downloads if the Census key is missing

In [ ]:
# Get pems data. Change the directory folder as per your requirement
osmbuilder.scrape_pems_data(station_metadata_option,
                    station_hr_data_option,
                    req_format,
                    year1=year_station_hr_data,
                    year2=year_station_metadata,
                    district_id=district_id, dir=folder_path)

In [ ]:
# Get open street map data
#37.3544,-121.9551 for Santa Clara. 
#37.3544,-121.9131 use this one shifted location for Santa Clara
#lat: 36.7394421, lon: -119.78483 for Fresno
osmbuilder.retrieve_graph_from_osm(area_of_interest, lat=lat, lon=lon, 
                                   width_km=width_km, height_km=height_km, dir=os.path.join(folder_path,'output'))

In [ ]:
osmbuilder.retrieve_graph_from_osm(area_of_interest, lat=lat, lon=lon, 
                                   width_km=width_km+5, height_km=height_km+5,dir=os.path.join(folder_path,'output/area_expansion'))

In [ ]:
# Map traffic counters to links
osmbuilder.traffic_counter_to_link(station_metadata_file = station_metadata_file,
                            area_gdf_file = "output/area_expansion/selected_area_gdf.shp",
                            nodes_file = "output/area_expansion/nodes_data.csv",
                            links_df_file = "output/area_expansion/link_list.geojson",
                            dist_threshold = 30,
                            counter_link_joined_data_file = "output/area_expansion/station_links_joined_data.geojson",
                            dir=folder_path)

In [ ]:
# Map traffic counters to links
osmbuilder.traffic_counter_to_link(station_metadata_file = station_metadata_file, 
                            area_gdf_file = "output/selected_area_gdf.shp",
                            nodes_file = "output/nodes_data.csv",
                            links_df_file = "output/link_list.geojson",
                            dist_threshold = 30,
                            counter_link_joined_data_file = "output/station_links_joined_data.geojson",
                            dir=folder_path)

In [ ]:
osmbuilder.traffic_counter_to_link_verify(counter_link_joined_data_file='output/station_links_joined_data.geojson',
                                          counter_link_joined_data_file_5='output/area_expansion/station_links_joined_data.geojson',
                                          dir=folder_path)

In [ ]:
# Get hourly data for matched links
osmbuilder.traffic_counter_hour_data_by_link(station_hour_data_file=station_hour_data_file,
                                             dir=folder_path)

In [ ]:
# Generate virtual links with taz or ct
# osmbuilder.generate_virtual_links_with_ct(dir='./examples/notebooks/data/')
# Needs the TAZ shapefile in <folder_path>/input/TAZ_Shap_Files (see README, Gather data)
osmbuilder.generate_virtual_links_with_taz(dir=folder_path)

In [ ]:
# Combine all nodes and links
osmbuilder.combine_nodes_n_links(dir=folder_path)

In [ ]:
# Prepare hourly data for all links in networks
osmbuilder.link_specific_null_values(dir=folder_path)

In [ ]:
# Census tracts with population (ACS 5-year B09019_001E), built once; needs CENSUS_API_KEY
counties_path = os.path.join(folder_path, counties_shp)
if not os.path.exists(counties_path):
    import pandas as pd
    import geopandas as gpd
    from census import Census
    acs = pd.DataFrame(Census(osmbuilder._credential('CENSUS_API_KEY')).acs5.state_county_tract(
        fields=('NAME', 'B09019_001E'), state_fips='06', county_fips='*', tract='*', year=2018))
    acs['GEOID'] = acs['state'] + acs['county'] + acs['tract']
    tracts = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2018/TRACT/tl_2018_06_tract.zip')
    tracts = tracts[['GEOID', 'geometry']].to_crs('EPSG:4326')
    tracts = tracts.merge(acs.rename(columns={'B09019_001E': 'total_hh'}), on='GEOID')
    os.makedirs(os.path.dirname(counties_path), exist_ok=True)
    tracts.to_file(counties_path)
print(counties_path)

In [ ]:
# Create prior ref_g values 
# osmbuilder.prior_data_reference_q_ct(dir='./examples/notebooks/data/')
osmbuilder.prior_data_reference_q_taz(county=county,
                                      counties_shp=counties_shp,
                                      dir=folder_path)